# LeetCode 889 — Construct Binary Tree from Preorder and Postorder Traversal bằng Python

Link đề: https://leetcode.com/problems/construct-binary-tree-from-preorder-and-postorder-traversal/ · Độ khó: **Medium** · Chủ đề: Tree (cây), Array (mảng), Hash Table (bảng băm), Divide and Conquer (chia để trị), Binary Tree (cây nhị phân)

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
        ...
```

Ghi chú: khác với "preorder + inorder", **preorder + postorder không xác định duy nhất** một cây khi có nút chỉ có **một con** (con đó có thể là con trái hoặc con phải mà hai dãy duyệt vẫn như nhau). Đề cho phép trả về **bất kỳ** cây hợp lệ, nên khi kiểm thử ta **tự dựng `TreeNode`**, rồi kiểm tra bằng hàm "preorder và postorder của cây trả về khớp hai dãy đầu vào" chứ không so với một cây cố định.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho hai dãy duyệt **tiền thứ tự** (gốc, trái, phải) và **hậu thứ tự** (trái, phải, gốc) của một cây nhị phân có các giá trị **khác nhau**; dựng lại một cây có hai dãy duyệt đó.
2. Nắm các công cụ Python cần thiết: lớp `TreeNode`, đệ quy trên **đoạn** `[a, b]` của hai mảng, bảng băm `pos = {giá trị: chỉ số trong postorder}`, ngăn xếp tường minh, và biến `nonlocal` làm con trỏ đọc.
3. Hiểu nguyên lý: gốc là phần tử **đầu** của preorder và **cuối** của postorder; phần tử **kế tiếp** trong preorder là gốc của cây con trái (nếu có), và vị trí của nó trong postorder cho biết **kích thước** cây con trái.
4. Đi qua 4 cách: thử mọi kích thước cây con trái (quay lui), chia để trị với bảng băm (cách nộp chính), ngăn xếp một lượt, và đệ quy với hai con trỏ đọc; đối chiếu bằng oracle **liệt kê mọi hình dạng cây** và hàm kiểm tra hợp lệ.
5. Tránh các lỗi kinh điển: so sánh với một đáp án cố định (bài có nhiều đáp án), sai số nút cây con trái lệch 1, thiếu trường hợp cơ sở một nút, và coi phần tử kế tiếp của preorder là gốc cây con **phải**.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `preorder` và `postorder`, trong đó `preorder` là duyệt tiền thứ tự và `postorder` là duyệt hậu thứ tự của **cùng một** cây nhị phân gồm các giá trị **khác nhau**. Hãy dựng lại và trả về cây nhị phân đó.

Nếu có nhiều đáp án, có thể trả về **bất kỳ** đáp án nào.

**Ví dụ 1**

- Input: `preorder = [1,2,4,5,3,6,7]`, `postorder = [4,5,2,6,7,3,1]`
- Output: `[1,2,3,4,5,6,7]` (cây hoàn chỉnh 7 nút, dạng duyệt theo mức)

**Ví dụ 2**

- Input: `preorder = [1]`, `postorder = [1]`
- Output: `[1]`

**Ràng buộc:**

- `1 <= preorder.length <= 30`
- `1 <= preorder[i] <= preorder.length`, các giá trị của `preorder` **khác nhau**
- `postorder.length == preorder.length`
- `1 <= postorder[i] <= postorder.length`, các giá trị của `postorder` khác nhau
- Đảm bảo hai mảng là tiền thứ tự và hậu thứ tự của cùng một cây nhị phân.

In [1]:
from typing import List, Optional, Tuple
from itertools import permutations
from functools import lru_cache
import random, time, sys

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(vals: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách theo mức kiểu LeetCode (None là nút trống)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

def tree_to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Xuất cây theo mức kiểu LeetCode (bỏ None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def preorder_of(root: Optional[TreeNode]) -> List[int]:
    return [] if root is None else [root.val] + preorder_of(root.left) + preorder_of(root.right)

def postorder_of(root: Optional[TreeNode]) -> List[int]:
    return [] if root is None else postorder_of(root.left) + postorder_of(root.right) + [root.val]

def is_valid(root: Optional[TreeNode], pre: List[int], post: List[int]) -> bool:
    """Cây trả về hợp lệ nếu preorder và postorder của nó đúng bằng hai dãy đầu vào."""
    return preorder_of(root) == list(pre) and postorder_of(root) == list(post)

examples = [
    ([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1], [1, 2, 3, 4, 5, 6, 7]),
    ([1], [1], [1]),
]
for pre, post, out in examples:
    t = build_tree(out)
    print(f"preorder={pre} postorder={post} -> cây {out}: hợp lệ? {is_valid(t, pre, post)}")
    assert is_valid(t, pre, post)

preorder=[1, 2, 4, 5, 3, 6, 7] postorder=[4, 5, 2, 6, 7, 3, 1] -> cây [1, 2, 3, 4, 5, 6, 7]: hợp lệ? True
preorder=[1] postorder=[1] -> cây [1]: hợp lệ? True


## 3. Kiến thức nền

### 3.1. Ba kiểu duyệt cây

- **Tiền thứ tự (preorder):** gốc, rồi cây con trái, rồi cây con phải.
- **Trung thứ tự (inorder):** trái, gốc, phải (không dùng ở bài này).
- **Hậu thứ tự (postorder):** trái, phải, rồi gốc.

Trong preorder, mỗi cây con chiếm một **đoạn liên tiếp** bắt đầu bằng gốc của nó; trong postorder, mỗi cây con chiếm một **đoạn liên tiếp** kết thúc bằng gốc của nó.

In [2]:
t = build_tree([1, 2, 3, 4, 5, 6, 7])
print("preorder :", preorder_of(t))
print("postorder:", postorder_of(t))
print("khôi phục danh sách theo mức:", tree_to_list(t))

preorder : [1, 2, 4, 5, 3, 6, 7]
postorder: [4, 5, 2, 6, 7, 3, 1]
khôi phục danh sách theo mức: [1, 2, 3, 4, 5, 6, 7]


### 3.2. Cấu trúc đoạn của hai dãy

Với cây có gốc `r`, cây con trái có `m` nút, cây con phải có `k` nút (`n = 1 + m + k`):

```
preorder  = [ r | cây trái (m nút) | cây phải (k nút) ]
postorder = [ cây trái (m nút) | cây phải (k nút) | r ]
```

Nếu biết `m` thì có thể cắt cả hai dãy thành các đoạn con tương ứng và đệ quy. Vấn đề chỉ là **tìm `m`**.

### 3.3. Tìm kích thước cây con trái nhờ phần tử kế tiếp của preorder

Nếu cây có cây con trái thì phần tử **ngay sau gốc** trong preorder là **gốc của cây con trái**, gọi là `L`. Trong postorder, `L` đứng ở **cuối đoạn của cây con trái**, nên nếu `L` ở chỉ số `i` (tính trong cả mảng postorder) và đoạn postorder của cây đang xét bắt đầu ở `c`, thì cây con trái có `m = i - c + 1` nút.

In [3]:
pre = [1, 2, 4, 5, 3, 6, 7]
post = [4, 5, 2, 6, 7, 3, 1]
pos = {x: i for i, x in enumerate(post)}
L = pre[1]
m = pos[L] - 0 + 1
print("gốc =", pre[0], "| gốc cây con trái L =", L, "| vị trí trong postorder =", pos[L], "| số nút cây con trái m =", m)
print("đoạn preorder của cây trái:", pre[1:1 + m], "| đoạn postorder:", post[:m])
print("đoạn preorder của cây phải:", pre[1 + m:], "| đoạn postorder:", post[m:-1])

gốc = 1 | gốc cây con trái L = 2 | vị trí trong postorder = 2 | số nút cây con trái m = 3
đoạn preorder của cây trái: [2, 4, 5] | đoạn postorder: [4, 5, 2]
đoạn preorder của cây phải: [3, 6, 7] | đoạn postorder: [6, 7, 3]


### 3.4. Vì sao đáp án **không duy nhất**

Nếu nút `v` chỉ có **một** con `c`: preorder có `... v c ...`, postorder có `... c v ...` dù `c` là con trái hay con phải. Hai cây (con trái và con phải) cho cùng cặp dãy. Do đó mỗi nút có đúng một con đóng góp một hệ số `2` vào số cây hợp lệ.

In [4]:
a = build_tree([1, 2])                 # 2 là con trái của 1
b = build_tree([1, None, 2])           # 2 là con phải của 1
print("cây a:", tree_to_list(a), "pre", preorder_of(a), "post", postorder_of(a))
print("cây b:", tree_to_list(b), "pre", preorder_of(b), "post", postorder_of(b))
assert preorder_of(a) == preorder_of(b) and postorder_of(a) == postorder_of(b)

cây a: [1, 2] pre [1, 2] post [2, 1]
cây b: [1, None, 2] pre [1, 2] post [2, 1]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Với ví dụ 1: `preorder = [1,2,4,5,3,6,7]`, `postorder = [4,5,2,6,7,3,1]`.

1. Gốc là `1` (đầu của preorder, cuối của postorder).
2. Phần tử kế tiếp trong preorder là `2`: gốc cây con trái. Trong postorder, `2` ở chỉ số `2` nên cây con trái có `2 - 0 + 1 = 3` nút: preorder `[2,4,5]`, postorder `[4,5,2]`.
3. Phần còn lại: cây con phải có preorder `[3,6,7]`, postorder `[6,7,3]`.
4. Làm lại cho `[2,4,5]` / `[4,5,2]`: gốc `2`, kế tiếp `4`, `4` ở chỉ số `0` của đoạn nên cây trái có `1` nút; cây phải là `[5]`.
5. Tương tự `[3,6,7]` / `[6,7,3]`: gốc `3`, trái `6`, phải `7`.

Kết quả: cây hoàn chỉnh `[1,2,3,4,5,6,7]`.

## 5. Cách 1 — Brute Force: thử mọi kích thước cây con trái

**Ý tưởng.** Không dùng mẹo "phần tử kế tiếp": thử **mọi** kích thước `m = 0..n-1` cho cây con trái. Một kích thước `m` chỉ có thể đúng nếu **tập giá trị** của `preorder[1:1+m]` bằng tập của `postorder[:m]` (hai đoạn cùng mô tả cây con trái). Với mỗi `m` thỏa điều kiện, đệ quy dựng cây trái và cây phải; nếu cả hai dựng được thì trả về, nếu không thì **quay lui** thử `m` khác.

In [5]:
def construct_brute(pre: List[int], post: List[int]) -> Optional[TreeNode]:
    n = len(pre)
    if n == 0:
        return None
    root = TreeNode(pre[0])
    if n == 1:
        return root if post[0] == pre[0] else None
    if post[-1] != pre[0]:
        return None
    for m in range(n):                                   # kích thước cây con trái
        if set(pre[1:1 + m]) != set(post[:m]):
            continue
        left = construct_brute(pre[1:1 + m], post[:m])
        if m > 0 and left is None:
            continue
        right = construct_brute(pre[1 + m:], post[m:-1])
        if n - 1 - m > 0 and right is None:
            continue
        root.left, root.right = left, right
        return root
    return None

for pre, post, out in examples:
    t = construct_brute(pre, post)
    print(pre, post, "->", tree_to_list(t))
    assert is_valid(t, pre, post)

[1, 2, 4, 5, 3, 6, 7] [4, 5, 2, 6, 7, 3, 1] -> [1, 2, 3, 4, 5, 6, 7]
[1] [1] -> [1]


**Độ phức tạp và nút thắt.** Mỗi mức thử `n` kích thước và so sánh tập `O(n)`: tệ nhất `O(n^2)` mỗi mức, và với cây "chuỗi" (mỗi nút một con) có tới `2^(n-1)` cây hợp lệ nếu liệt kê hết. Xem số cây hợp lệ tăng thế nào (đếm bằng cách liệt kê mọi cây khớp hai dãy):

In [6]:
def all_trees(pre: List[int], post: List[int]) -> List[tuple]:
    """Liệt kê mọi cây (dạng tuple lồng nhau) khớp pre/post, bằng cách thử mọi kích thước cây trái."""
    n = len(pre)
    if n == 0:
        return [None]
    if n == 1:
        return [(pre[0], None, None)] if post[0] == pre[0] else []
    if post[-1] != pre[0]:
        return []
    res = []
    for m in range(n):
        if set(pre[1:1 + m]) != set(post[:m]):
            continue
        for l in all_trees(pre[1:1 + m], post[:m]):
            for r in all_trees(pre[1 + m:], post[m:-1]):
                res.append((pre[0], l, r))
    return res

for n in range(1, 13):
    pre = list(range(1, n + 1))
    post = pre[::-1]                      # cây "chuỗi": mọi nút trừ lá có đúng một con
    t0 = time.perf_counter()
    cnt = len(all_trees(pre, post))
    print(f"n={n:2}: {cnt:5} cây hợp lệ ({2 ** (n - 1)} = 2^(n-1)), {1000 * (time.perf_counter() - t0):8.2f} ms")

n= 1:     1 cây hợp lệ (1 = 2^(n-1)),     0.01 ms
n= 2:     2 cây hợp lệ (2 = 2^(n-1)),     0.01 ms
n= 3:     4 cây hợp lệ (4 = 2^(n-1)),     0.02 ms
n= 4:     8 cây hợp lệ (8 = 2^(n-1)),     0.03 ms
n= 5:    16 cây hợp lệ (16 = 2^(n-1)),     0.06 ms
n= 6:    32 cây hợp lệ (32 = 2^(n-1)),     0.13 ms
n= 7:    64 cây hợp lệ (64 = 2^(n-1)),     0.31 ms
n= 8:   128 cây hợp lệ (128 = 2^(n-1)),     0.85 ms
n= 9:   256 cây hợp lệ (256 = 2^(n-1)),     1.24 ms
n=10:   512 cây hợp lệ (512 = 2^(n-1)),     2.33 ms
n=11:  1024 cây hợp lệ (1024 = 2^(n-1)),     3.66 ms
n=12:  2048 cây hợp lệ (2048 = 2^(n-1)),    13.53 ms


Số cây hợp lệ tăng gấp đôi mỗi nút. Dù chỉ cần **một** cây, việc thử mọi kích thước cho mỗi nút là lãng phí: có cách **xác định ngay** kích thước cây con trái.

## 6. Ý tưởng thuật toán / bước đột phá

**Phần tử kế tiếp của preorder cho biết gốc cây con trái.** Nếu đoạn hiện tại có hơn một nút, ta **chọn** quy ước: `preorder[a + 1]` là gốc cây con **trái** (nếu thực tế nó là cây con phải của một nút chỉ có một con thì cây dựng ra vẫn có cùng hai dãy duyệt, vì hai trường hợp cho cùng cặp dãy). Khi đó:

- tra vị trí `i` của `preorder[a + 1]` trong postorder (bảng băm `pos`, `O(1)`);
- cây con trái có `m = i - c + 1` nút (`c` là đầu đoạn postorder hiện tại).

Mỗi nút chỉ xử lý một lần ⇒ `O(n)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Chia để trị trên đoạn `[a, b]`, `[c, d]` với bảng băm `pos` (nộp) |
| 3 | Ngăn xếp một lượt: duyệt preorder, dùng postorder để biết khi nào "đóng" nút |
| 4 | Đệ quy với hai con trỏ đọc `i` (preorder), `j` (postorder), không cắt mảng |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Chia để trị với bảng băm (cách nộp chính)

**Giải thích từng đoạn.**

- `pos = {x: i for i, x in enumerate(postorder)}`: vị trí trong postorder.
- `dfs(a, b, c, d)`: dựng cây từ `preorder[a..b]` và `postorder[c..d]`.
  - `a > b`: đoạn rỗng, trả `None`.
  - `root = TreeNode(preorder[a])`; nếu `a == b` (một nút) trả `root`.
  - Ngược lại `i = pos[preorder[a + 1]]`, `m = i - c + 1`:
    - cây trái: `dfs(a + 1, a + m, c, i)`;
    - cây phải: `dfs(a + m + 1, b, i + 1, d - 1)`.

In [7]:
def construct_dc(pre: List[int], post: List[int]) -> Optional[TreeNode]:
    pos = {x: i for i, x in enumerate(post)}

    def dfs(a: int, b: int, c: int, d: int) -> Optional[TreeNode]:
        if a > b:
            return None
        root = TreeNode(pre[a])
        if a == b:
            return root
        i = pos[pre[a + 1]]
        m = i - c + 1
        root.left = dfs(a + 1, a + m, c, i)
        root.right = dfs(a + m + 1, b, i + 1, d - 1)
        return root

    return dfs(0, len(pre) - 1, 0, len(post) - 1)

for pre, post, out in examples:
    t = construct_dc(pre, post)
    print(pre, post, "->", tree_to_list(t))
    assert is_valid(t, pre, post) and tree_to_list(t) == out

[1, 2, 4, 5, 3, 6, 7] [4, 5, 2, 6, 7, 3, 1] -> [1, 2, 3, 4, 5, 6, 7]
[1] [1] -> [1]


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
        pos = {x: i for i, x in enumerate(postorder)}

        def dfs(a: int, b: int, c: int, d: int) -> Optional[TreeNode]:
            if a > b:
                return None
            root = TreeNode(preorder[a])
            if a == b:
                return root
            i = pos[preorder[a + 1]]
            m = i - c + 1
            root.left = dfs(a + 1, a + m, c, i)
            root.right = dfs(a + m + 1, b, i + 1, d - 1)
            return root

        return dfs(0, len(preorder) - 1, 0, len(postorder) - 1)

sol_dc = Solution().constructFromPrePost
for pre, post, out in examples:
    assert is_valid(sol_dc(pre, post), pre, post)
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n)` thời gian (mỗi nút một lần, mỗi lần `O(1)`), `O(n)` bộ nhớ cho `pos` và chiều sâu đệ quy tối đa `O(n)`.

---

### 7.2. Cách 3 — Ngăn xếp một lượt

**Ý tưởng khác.** Duyệt `preorder` từ phần tử thứ hai, giữ **ngăn xếp** các nút "còn mở" (đã tạo nhưng chưa biết đã đủ con chưa), và dùng con trỏ `j` trên `postorder`:

- Nút đỉnh ngăn xếp có `val == postorder[j]` nghĩa là subtree của nó **đã xong** (vì postorder in nút khi cả hai cây con đã in xong); ta `pop` và tăng `j`.
- Sau khi đóng các nút xong, nút mới `node` là con của nút ở đỉnh: nếu đỉnh chưa có con trái thì gán trái, ngược lại gán phải; rồi đẩy `node` vào ngăn xếp.

**Giải thích từng đoạn.** Quy ước "chưa có con trái thì gán trái" chính là chọn con trái cho nút chỉ có một con (như Cách 2).

In [9]:
def construct_stack(pre: List[int], post: List[int]) -> Optional[TreeNode]:
    root = TreeNode(pre[0])
    stack = [root]
    j = 0
    for v in pre[1:]:
        node = TreeNode(v)
        while stack[-1].val == post[j]:
            stack.pop()
            j += 1
        if stack[-1].left is None:
            stack[-1].left = node
        else:
            stack[-1].right = node
        stack.append(node)
    return root

for pre, post, out in examples:
    t = construct_stack(pre, post)
    print(pre, post, "->", tree_to_list(t))
    assert is_valid(t, pre, post)

[1, 2, 4, 5, 3, 6, 7] [4, 5, 2, 6, 7, 3, 1] -> [1, 2, 3, 4, 5, 6, 7]
[1] [1] -> [1]


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
        root = TreeNode(preorder[0])
        stack = [root]
        j = 0
        for v in preorder[1:]:
            node = TreeNode(v)
            while stack[-1].val == postorder[j]:
                stack.pop()
                j += 1
            if stack[-1].left is None:
                stack[-1].left = node
            else:
                stack[-1].right = node
            stack.append(node)
        return root

sol_stack = Solution().constructFromPrePost
for pre, post, out in examples:
    assert is_valid(sol_stack(pre, post), pre, post)
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n)` thời gian (mỗi nút đẩy và lấy ra tối đa một lần), `O(n)` bộ nhớ cho ngăn xếp (và kết quả).

---

### 7.3. Cách 4 — Đệ quy hai con trỏ đọc

**Ý tưởng khác.** Hai con trỏ toàn cục `i` (vị trí đọc trong preorder) và `j` (trong postorder). Hàm `build()` tạo nút gốc `preorder[i]` rồi `i += 1`. Nếu **gốc chưa trùng** `postorder[j]` thì cây con trái chưa xong ⇒ `root.left = build()`; nếu gốc vẫn chưa trùng `postorder[j]` thì `root.right = build()`. Khi gốc trùng `postorder[j]` thì subtree đã đọc xong ⇒ `j += 1`, trả `root`.

**Giải thích từng đoạn.** Không cắt mảng, không bảng băm: dùng chính thứ tự đọc của hai dãy để biết khi nào subtree kết thúc.

In [11]:
def construct_pointers(pre: List[int], post: List[int]) -> Optional[TreeNode]:
    i = j = 0

    def build() -> TreeNode:
        nonlocal i, j
        root = TreeNode(pre[i])
        i += 1
        if root.val != post[j]:
            root.left = build()
        if root.val != post[j]:
            root.right = build()
        j += 1
        return root

    return build()

for pre, post, out in examples:
    t = construct_pointers(pre, post)
    print(pre, post, "->", tree_to_list(t))
    assert is_valid(t, pre, post)

[1, 2, 4, 5, 3, 6, 7] [4, 5, 2, 6, 7, 3, 1] -> [1, 2, 3, 4, 5, 6, 7]
[1] [1] -> [1]


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
        i = j = 0

        def build() -> TreeNode:
            nonlocal i, j
            root = TreeNode(preorder[i])
            i += 1
            if root.val != postorder[j]:
                root.left = build()
            if root.val != postorder[j]:
                root.right = build()
            j += 1
            return root

        return build()

sol_ptr = Solution().constructFromPrePost
for pre, post, out in examples:
    assert is_valid(sol_ptr(pre, post), pre, post)
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n)` thời gian, `O(h)` bộ nhớ ngoài kết quả (`h` là chiều cao cây, tới `n`).

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 1

In từng lần gọi `dfs(a, b, c, d)` với đoạn preorder, đoạn postorder, vị trí `i` của gốc cây trái và kích thước `m`.

In [13]:
def trace_dc(pre, post):
    pos = {x: i for i, x in enumerate(post)}

    def dfs(a, b, c, d, depth=0):
        pad = "  " * depth
        if a > b:
            print(f"{pad}dfs({a},{b},{c},{d}): rỗng -> None")
            return None
        root = TreeNode(pre[a])
        if a == b:
            print(f"{pad}dfs({a},{b},{c},{d}): một nút {root.val}")
            return root
        i = pos[pre[a + 1]]
        m = i - c + 1
        print(f"{pad}dfs({a},{b},{c},{d}): pre={pre[a:b + 1]} post={post[c:d + 1]} gốc={root.val}, gốc trái={pre[a + 1]} ở post[{i}] -> m={m}")
        root.left = dfs(a + 1, a + m, c, i, depth + 1)
        root.right = dfs(a + m + 1, b, i + 1, d - 1, depth + 1)
        return root

    return dfs(0, len(pre) - 1, 0, len(post) - 1)

t = trace_dc([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1])
assert tree_to_list(t) == [1, 2, 3, 4, 5, 6, 7]

dfs(0,6,0,6): pre=[1, 2, 4, 5, 3, 6, 7] post=[4, 5, 2, 6, 7, 3, 1] gốc=1, gốc trái=2 ở post[2] -> m=3
  dfs(1,3,0,2): pre=[2, 4, 5] post=[4, 5, 2] gốc=2, gốc trái=4 ở post[0] -> m=1
    dfs(2,2,0,0): một nút 4
    dfs(3,3,1,1): một nút 5
  dfs(4,6,3,5): pre=[3, 6, 7] post=[6, 7, 3] gốc=3, gốc trái=6 ở post[3] -> m=1
    dfs(5,5,3,3): một nút 6
    dfs(6,6,4,4): một nút 7


### 8.2. Cách 3 (ngăn xếp) cho ví dụ 1

`preorder = [1,2,4,5,3,6,7]`, `postorder = [4,5,2,6,7,3,1]`. Bảng sau cho thấy ngăn xếp (giá trị) và con trỏ `j` sau mỗi bước.

In [14]:
def trace_stack(pre, post):
    root = TreeNode(pre[0])
    stack = [root]
    j = 0
    print(f"khởi tạo: stack={[x.val for x in stack]} j={j}")
    for v in pre[1:]:
        node = TreeNode(v)
        popped = []
        while stack[-1].val == post[j]:
            popped.append(stack.pop().val)
            j += 1
        if stack[-1].left is None:
            stack[-1].left = node
            side = "trái"
        else:
            stack[-1].right = node
            side = "phải"
        parent = stack[-1].val
        stack.append(node)
        print(f"đọc {v}: đóng {popped} -> {v} là con {side} của {parent}; stack={[x.val for x in stack]} j={j}")
    return root

t = trace_stack([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1])
assert tree_to_list(t) == [1, 2, 3, 4, 5, 6, 7]

khởi tạo: stack=[1] j=0
đọc 2: đóng [] -> 2 là con trái của 1; stack=[1, 2] j=0
đọc 4: đóng [] -> 4 là con trái của 2; stack=[1, 2, 4] j=0
đọc 5: đóng [4] -> 5 là con phải của 2; stack=[1, 2, 5] j=1
đọc 3: đóng [5, 2] -> 3 là con phải của 1; stack=[1, 3] j=3
đọc 6: đóng [] -> 6 là con trái của 3; stack=[1, 3, 6] j=3
đọc 7: đóng [6] -> 7 là con phải của 3; stack=[1, 3, 7] j=4


### 8.3. Cách 4 (hai con trỏ) cho cây một con: `preorder = [1,2,3]`, `postorder = [3,2,1]`

`build()` gốc `1`: `1 != post[0] = 3` ⇒ `left = build()`: gốc `2`, `2 != 3` ⇒ `left = build()`: gốc `3`, `3 == post[0]` ⇒ không có con, `j = 1`; quay lại gốc `2`: `2 == post[1] = 2` ⇒ không có con phải, `j = 2`; quay lại gốc `1`: `1 == post[2]` ⇒ không có con phải, `j = 3`. Cây chuỗi trái: `[1, 2, None, 3]`.

In [15]:
t = construct_pointers([1, 2, 3], [3, 2, 1])
print(tree_to_list(t))
assert tree_to_list(t) == [1, 2, None, 3]

[1, 2, None, 3]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (chia để trị với bảng băm): ý tưởng "phần tử kế tiếp của preorder cho kích thước cây con trái" rõ ràng, `O(n)` và dễ chứng minh.

```python
class Solution:
    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
        pos = {x: i for i, x in enumerate(postorder)}

        def dfs(a: int, b: int, c: int, d: int) -> Optional[TreeNode]:
            if a > b:
                return None
            root = TreeNode(preorder[a])
            if a == b:
                return root
            i = pos[preorder[a + 1]]
            m = i - c + 1
            root.left = dfs(a + 1, a + m, c, i)
            root.right = dfs(a + m + 1, b, i + 1, d - 1)
            return root

        return dfs(0, len(preorder) - 1, 0, len(postorder) - 1)
```

In [16]:
class Solution:
    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
        pos = {x: i for i, x in enumerate(postorder)}

        def dfs(a: int, b: int, c: int, d: int) -> Optional[TreeNode]:
            if a > b:
                return None
            root = TreeNode(preorder[a])
            if a == b:
                return root
            i = pos[preorder[a + 1]]
            m = i - c + 1
            root.left = dfs(a + 1, a + m, c, i)
            root.right = dfs(a + m + 1, b, i + 1, d - 1)
            return root

        return dfs(0, len(preorder) - 1, 0, len(postorder) - 1)

sol = Solution()
for pre, post, out in examples:
    t = sol.constructFromPrePost(pre, post)
    print(pre, post, "->", tree_to_list(t), "hợp lệ" if is_valid(t, pre, post) else "SAI")
    assert is_valid(t, pre, post)
    assert tree_to_list(t) == out          # ở hai ví dụ này cây là duy nhất

[1, 2, 4, 5, 3, 6, 7] [4, 5, 2, 6, 7, 3, 1] -> [1, 2, 3, 4, 5, 6, 7] hợp lệ
[1] [1] -> [1] hợp lệ


## 10. Vì sao đúng

**(a) Cấu trúc đoạn.** Với cây gốc `r`, cây trái `m` nút, cây phải `k` nút, `preorder = [r] + P(trái) + P(phải)` và `postorder = Q(trái) + Q(phải) + [r]`. Vậy gốc `r = preorder[a] = postorder[d]` và mọi cây con ứng với một đoạn liên tiếp của mỗi dãy, nên chia đoạn là đúng.

**(b) Cách 2 (nút có hai con).** Nếu gốc có cây trái không rỗng, phần tử `preorder[a + 1]` là gốc của cây trái. Trong `postorder`, gốc của cây trái đứng **cuối** đoạn `Q(trái)`, ở chỉ số `i`, nên `Q(trái) = postorder[c..i]` có `m = i - c + 1` nút; `P(trái) = preorder[a + 1..a + m]`; phần còn lại thuộc cây phải: `preorder[a + m + 1..b]` và `postorder[i + 1..d - 1]`. Đệ quy cho đúng cây.

**(c) Nút chỉ có một con.** Giả sử `v` chỉ có một con `c`. Dù `c` là con trái hay phải, preorder của subtree là `v, P(c)` và postorder là `Q(c), v`. Cách 2 lấy `c = preorder[a + 1]` và tra thấy ở **cuối** `postorder[c..d-1]`, nên `m = d - c` nút và cây phải rỗng: ta dựng `c` làm **con trái**. Cây này có cùng hai dãy duyệt, nên là đáp án hợp lệ (cây khác với đáp án mẫu nếu mẫu để `c` ở bên phải, nhưng đề chấp nhận mọi đáp án).

**(d) Số cây hợp lệ.** Mỗi nút có đúng một con cho hai lựa chọn (con trái hoặc con phải) mà không đổi hai dãy duyệt, các nút khác bị xác định ⇒ số cây hợp lệ là `2^s` với `s` là số nút có đúng một con. Mục 15 kiểm chứng điều này.

**(e) Cách 3 và 4.** Cả hai đều dựng đúng cây theo quy ước ở (c): Cách 3 gán nút mới làm con trái nếu nút đỉnh chưa có con trái; Cách 4 chỉ dựng cây phải nếu gốc vẫn chưa trùng `postorder[j]` sau khi dựng cây trái, tức là sau cây trái vẫn còn nút chưa đóng. Hai quy tắc này trùng với "chọn con trái trước".

**Kiểm chứng bằng máy.** Oracle **độc lập với mọi lời giải**: liệt kê **mọi hình dạng cây** nhị phân có `n` nút (số Catalan), gắn nhãn theo `preorder` đã cho, tính postorder và giữ các hình có postorder đúng bằng `postorder` đã cho. Mọi cây do lời giải dựng phải (1) thỏa `is_valid`, (2) thuộc tập hình dạng của oracle.

In [17]:
@lru_cache(maxsize=None)
def shapes(n: int) -> tuple:
    """Mọi hình dạng cây nhị phân có n nút (không gán giá trị): None hoặc (hình_trái, hình_phải)."""
    if n == 0:
        return (None,)
    out = []
    for k in range(n):
        for l in shapes(k):
            for r in shapes(n - 1 - k):
                out.append((l, r))
    return tuple(out)

def label_with_preorder(shape, it):
    if shape is None:
        return None
    val = next(it)
    l = label_with_preorder(shape[0], it)
    r = label_with_preorder(shape[1], it)
    return (val, l, r)

def post_of_labeled(t):
    return [] if t is None else post_of_labeled(t[1]) + post_of_labeled(t[2]) + [t[0]]

def oracle_trees(pre: List[int], post: List[int]) -> set:
    """Tập mọi cây (dạng tuple lồng nhau) có preorder = pre và postorder = post, bằng cách thử mọi hình dạng."""
    res = set()
    for sh in shapes(len(pre)):
        t = label_with_preorder(sh, iter(pre))
        if post_of_labeled(t) == list(post):
            res.add(t)
    return res

def to_labeled(root: Optional[TreeNode]):
    return None if root is None else (root.val, to_labeled(root.left), to_labeled(root.right))

# Oracle liệt kê hình dạng khớp với oracle "thử mọi kích thước cây trái" (hai cách khác nhau)
rng = random.Random(889)
def random_shape_tree(rng, n):
    """Cây ngẫu nhiên n nút, nhãn là hoán vị của 1..n. Trả về (pre, post, tuple)."""
    def make(k):
        if k == 0:
            return None
        l = rng.randint(0, k - 1)
        return (make(l), make(k - 1 - l))
    sh = make(n)
    labels = list(range(1, n + 1))
    rng.shuffle(labels)
    # gán nhãn theo preorder: preorder = labels
    t = label_with_preorder(sh, iter(labels))
    return labels, post_of_labeled(t), t

for _ in range(150):
    n = rng.randint(1, 8)
    pre, post, t = random_shape_tree(rng, n)
    o = oracle_trees(pre, post)
    b = set(all_trees(pre, post))
    assert o == b and t in o, (pre, post)
print("oracle liệt kê hình dạng khớp oracle thử-mọi-kích-thước và luôn chứa cây gốc, trên 150 cây ngẫu nhiên")

oracle liệt kê hình dạng khớp oracle thử-mọi-kích-thước và luôn chứa cây gốc, trên 150 cây ngẫu nhiên


## 11. Time / Space Complexity

Gọi `n` là số nút.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi kích thước (quay lui) | tệ nhất `O(n^2)` mỗi mức × số lần thử lại (nhưng với đầu vào hợp lệ chỉ thử vài lần) | `O(n)` mỗi mức cộng bản sao mảng |
| 2. Chia để trị + `pos` | `O(n)` | `O(n)` |
| 3. Ngăn xếp | `O(n)` | `O(n)` |
| 4. Hai con trỏ | `O(n)` | `O(n)` |

Ràng buộc đề `n <= 30`; ta đo trên `n` lớn hơn (nhưng chiều sâu vừa phải) để thấy xu hướng:

In [18]:
def random_tree_nodes(rng, n):
    """Cây ngẫu nhiên n nút dựng lặp (không đệ quy), nhãn 1..n. Trả về gốc."""
    vals = list(range(1, n + 1))
    rng.shuffle(vals)
    root = TreeNode(vals[0])
    slots = [(root, "left"), (root, "right")]
    for v in vals[1:]:
        idx = rng.randrange(len(slots))
        slots[idx], slots[-1] = slots[-1], slots[idx]
        parent, side = slots.pop()
        node = TreeNode(v)
        setattr(parent, side, node)
        slots.append((node, "left"))
        slots.append((node, "right"))
    return root

def iter_pre(root):
    out, st = [], [root] if root else []
    while st:
        x = st.pop()
        out.append(x.val)
        if x.right: st.append(x.right)
        if x.left: st.append(x.left)
    return out

def iter_post(root):
    out, st = [], [root] if root else []
    while st:
        x = st.pop()
        out.append(x.val)
        if x.left: st.append(x.left)
        if x.right: st.append(x.right)
    return out[::-1]

rng = random.Random(11)
sys.setrecursionlimit(10000)
for n in (30, 300, 1000, 3000):
    root = random_tree_nodes(rng, n)
    pre, post = iter_pre(root), iter_post(root)
    row = []
    for name, fn in (("Cách 2", sol_dc), ("Cách 3", sol_stack), ("Cách 4", sol_ptr)):
        t0 = time.perf_counter()
        t = fn(pre, post)
        row.append(f"{name}: {1000 * (time.perf_counter() - t0):7.2f} ms")
        assert iter_pre(t) == pre and iter_post(t) == post, (name, n)
    print(f"n={n:5}: " + " | ".join(row))
sys.setrecursionlimit(1000)

n=   30: Cách 2:    0.03 ms | Cách 3:    0.02 ms | Cách 4:    0.01 ms
n=  300: Cách 2:    0.18 ms | Cách 3:    0.17 ms | Cách 4:    0.13 ms
n= 1000: Cách 2:    0.87 ms | Cách 3:    0.56 ms | Cách 4:    0.51 ms
n= 3000: Cách 2:    2.26 ms | Cách 3:    1.82 ms | Cách 4:    1.62 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle hình dạng, test ngẫu nhiên có seed (cây ngẫu nhiên đúng ràng buộc: giá trị là hoán vị của `1..n`, `n <= 30`) và tính chất. Vì có nhiều đáp án đúng nên mọi kiểm tra dùng **`is_valid`** (preorder/postorder của cây trả về khớp đầu vào), không so cây cố định.

In [19]:
def solutions_list():
    return {
        "brute": construct_brute,
        "dc": construct_dc,
        "stack": construct_stack,
        "pointers": construct_pointers,
        "sol_dc": sol_dc,
        "sol_stack": sol_stack,
        "sol_ptr": sol_ptr,
        "final": Solution().constructFromPrePost,
    }

def run_all(pre, post, use_oracle=True):
    """Chạy mọi lời giải trên (pre, post); kiểm tra hợp lệ và (nếu n nhỏ) thuộc tập oracle. Trả về số cây hợp lệ của oracle (hoặc None)."""
    n = len(pre)
    oracle = oracle_trees(pre, post) if use_oracle and n <= 9 else None
    for name, fn in solutions_list().items():
        t = fn(list(pre), list(post))
        assert t is not None and is_valid(t, pre, post), (name, pre, post, tree_to_list(t))
        if oracle is not None:
            assert to_labeled(t) in oracle, (name, pre, post)
    return None if oracle is None else len(oracle)

# 12.1 ví dụ đề
for pre, post, out in examples:
    run_all(pre, post)
    assert tree_to_list(Solution().constructFromPrePost(pre, post)) == out
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng số cây hợp lệ viết tay, kèm lý do)
edge = [
    ([1], [1], 1),                                   # một nút
    ([1, 2], [2, 1], 2),                             # con trái hoặc con phải
    ([1, 2, 3], [2, 3, 1], 1),                       # hai con: duy nhất
    ([1, 2, 3], [3, 2, 1], 4),                       # chuỗi: mỗi nút (1, 2) một con -> 2^2
    ([1, 2, 3, 4], [4, 3, 2, 1], 8),                 # chuỗi dài 4: 2^3
    ([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1], 1),   # hoàn chỉnh 7 nút
    ([2, 1, 3], [1, 3, 2], 1),                       # nhãn không theo thứ tự
    ([3, 1, 2], [2, 1, 3], 4),                       # chuỗi nhãn tùy ý
    ([1, 2, 3, 4], [3, 4, 2, 1], 2),                 # 2 có hai con (3,4), 1 có một con (2): 2^1
    ([1, 2, 3, 4], [3, 2, 4, 1], 2),                 # 1 có hai con (2, 4), 2 có một con (3)
]
for pre, post, expected in edge:
    got = run_all(pre, post)
    assert got == expected, (pre, post, got, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 10 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_off_by_one(pre, post):                  # kích thước cây trái m = i - c (thiếu +1)
    pos = {x: i for i, x in enumerate(post)}

    def dfs(a, b, c, d):
        if a > b:
            return None
        root = TreeNode(pre[a])
        if a == b:
            return root
        i = pos[pre[a + 1]]
        m = i - c                                  # sai: phải là i - c + 1
        root.left = dfs(a + 1, a + m, c, i)
        root.right = dfs(a + m + 1, b, i + 1, d - 1)
        return root

    return dfs(0, len(pre) - 1, 0, len(post) - 1)

def wrong_no_base(pre, post):                     # thiếu trường hợp một nút (a == b): đọc pre[a + 1] ngoài đoạn
    pos = {x: i for i, x in enumerate(post)}

    def dfs(a, b, c, d):
        if a > b:
            return None
        root = TreeNode(pre[a])
        i = pos[pre[a + 1]]                        # IndexError khi a == len(pre) - 1
        m = i - c + 1
        root.left = dfs(a + 1, a + m, c, i)
        root.right = dfs(a + m + 1, b, i + 1, d - 1)
        return root

    return dfs(0, len(pre) - 1, 0, len(post) - 1)

def wrong_right_root(pre, post):                  # coi pre[a + 1] là gốc cây con PHẢI (kết thúc cuối post)
    pos = {x: i for i, x in enumerate(post)}

    def dfs(a, b, c, d):
        if a > b:
            return None
        root = TreeNode(pre[a])
        if a == b:
            return root
        i = pos[pre[a + 1]]
        m = d - 1 - i + 1                          # số nút của cây "phải" nếu pre[a + 1] là gốc của nó
        root.right = dfs(a + 1, a + m, i - m + 1, i)
        root.left = dfs(a + m + 1, b, c, i - m)
        return root

    return dfs(0, len(pre) - 1, 0, len(post) - 1)

def attempt(fn, pre, post):
    try:
        t = fn(pre, post)
    except Exception as e:
        return f"lỗi {type(e).__name__}"
    return "hợp lệ" if (t is not None and is_valid(t, pre, post)) else "SAI"

def random_instance(rng, n):
    pre, post, _ = random_shape_tree(rng, n)
    return pre, post

def find_counter(fn, tries=4000, seed=2):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 8)
        pre, post = random_instance(rng, n)
        if attempt(fn, pre, post) != "hợp lệ":
            return pre, post
    return None

counters = {}
for name, fn in (("sai số nút cây trái lệch 1", wrong_off_by_one), ("thiếu trường hợp một nút", wrong_no_base),
                 ("coi pre[a+1] là gốc cây phải", wrong_right_root)):
    c = find_counter(fn)
    assert c is not None, name
    counters[name] = c
    pre, post = c
    print(f"{name:32} pre={pre} post={post}: {attempt(fn, pre, post)}")
    run_all(pre, post)                               # các lời giải đúng không bị lỗi trên chính ca này
print("12.3 phản ví dụ: OK")

sai số nút cây trái lệch 1       pre=[8, 7, 3, 4, 2, 6, 5, 1] post=[6, 2, 1, 5, 4, 3, 7, 8]: SAI


thiếu trường hợp một nút         pre=[8, 7, 3, 4, 2, 6, 5, 1] post=[6, 2, 1, 5, 4, 3, 7, 8]: lỗi IndexError
coi pre[a+1] là gốc cây phải     pre=[8, 7, 3, 4, 2, 6, 5, 1] post=[6, 2, 1, 5, 4, 3, 7, 8]: lỗi RecursionError


12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: n <= 8, mọi cách khớp oracle hình dạng
rng = random.Random(4889)
total_trees = 0
for _ in range(1500):
    n = rng.randint(1, 8)
    pre, post, t = random_shape_tree(rng, n)
    cnt = run_all(pre, post)
    assert to_labeled(Solution().constructFromPrePost(pre, post)) in oracle_trees(pre, post)
    total_trees += cnt
print(f"12.4 ngẫu nhiên n <= 8 (1500 ca): OK, tổng số cây hợp lệ theo oracle {total_trees}")

# 12.5 ngẫu nhiên đúng ràng buộc đề: n <= 30 (kiểm tra bằng is_valid, không dùng oracle hình dạng)
rng = random.Random(5889)
for _ in range(600):
    n = rng.randint(1, 30)
    pre, post, t = random_shape_tree(rng, n)
    assert sorted(pre) == list(range(1, n + 1))
    run_all(pre, post, use_oracle=False)
print("12.5 ngẫu nhiên n <= 30 (600 ca): OK")

# 12.6 các dạng đặc biệt n <= 30: chuỗi, cây đầy đủ, nhãn đảo
special = {}
n = 30
special["chuỗi (trái hoặc phải, cùng cặp dãy)"] = (list(range(1, n + 1)), list(range(n, 0, -1)))
special["cây đầy đủ 15"] = ([1, 2, 4, 8, 9, 5, 10, 11, 3, 6, 12, 13, 7, 14, 15], [8, 9, 4, 10, 11, 5, 2, 12, 13, 6, 14, 15, 7, 3, 1])
special["nhãn đảo"] = (list(range(n, 0, -1)), list(range(1, n + 1)))
for name, (pre, post) in special.items():
    run_all(pre, post, use_oracle=False)
print("12.6 các dạng đặc biệt: OK", list(special))

# 12.7 tính chất: kết quả chỉ phụ thuộc vào (pre, post): đổi nhãn bằng một song ánh giữ nguyên tính hợp lệ
rng = random.Random(7889)
for _ in range(300):
    n = rng.randint(1, 12)
    pre, post, t = random_shape_tree(rng, n)
    perm = list(range(1, n + 1))
    rng.shuffle(perm)
    mp = {i + 1: perm[i] for i in range(n)}
    pre2, post2 = [mp[x] for x in pre], [mp[x] for x in post]
    tt = Solution().constructFromPrePost(pre2, post2)
    assert is_valid(tt, pre2, post2)
    if n <= 8:
        assert len(oracle_trees(pre, post)) == len(oracle_trees(pre2, post2))
print("12.7 bất biến theo đổi nhãn: OK")

12.4 ngẫu nhiên n <= 8 (1500 ca): OK, tổng số cây hợp lệ theo oracle 7793
12.5 ngẫu nhiên n <= 30 (600 ca): OK
12.6 các dạng đặc biệt: OK ['chuỗi (trái hoặc phải, cùng cặp dãy)', 'cây đầy đủ 15', 'nhãn đảo']


12.7 bất biến theo đổi nhãn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh với một đáp án cố định

Bài có **nhiều đáp án**. Cây của bạn đúng ngay cả khi khác đáp án mẫu. Cặp `preorder = [1,2]`, `postorder = [2,1]` có **hai** cây hợp lệ.

In [22]:
pre, post = [1, 2], [2, 1]
mine = Solution().constructFromPrePost(pre, post)
other = build_tree([1, None, 2])
print("cây của lời giải:", tree_to_list(mine), "| cây khác cũng hợp lệ:", tree_to_list(other))
print("cả hai hợp lệ?", is_valid(mine, pre, post), is_valid(other, pre, post))
print("hai danh sách theo mức có bằng nhau không?", tree_to_list(mine) == tree_to_list(other), "-> so với một đáp án cố định sẽ báo SAI oan")
assert is_valid(mine, pre, post) and is_valid(other, pre, post) and tree_to_list(mine) != tree_to_list(other)
print("số cây hợp lệ theo oracle:", len(oracle_trees(pre, post)))

cây của lời giải: [1, 2] | cây khác cũng hợp lệ: [1, None, 2]
cả hai hợp lệ? True True
hai danh sách theo mức có bằng nhau không? False -> so với một đáp án cố định sẽ báo SAI oan
số cây hợp lệ theo oracle: 2


Vì sao: preorder và postorder không phân biệt con trái/con phải của nút chỉ có một con. **Đúng:** kiểm tra bằng `is_valid` (hai dãy duyệt của cây trả về khớp đầu vào).

### Lỗi 2 — Kích thước cây con trái lệch 1

`i` là chỉ số của gốc cây con trái trong postorder (tính **bao gồm** chính nó), nên số nút cây trái là `i - c + 1`, không phải `i - c`.

In [23]:
pre, post = counters["sai số nút cây trái lệch 1"]
print("pre =", pre, "post =", post, "->", attempt(wrong_off_by_one, pre, post), "| bản đúng ->", attempt(Solution().constructFromPrePost, pre, post))
assert attempt(wrong_off_by_one, pre, post) != "hợp lệ"

pre = [8, 7, 3, 4, 2, 6, 5, 1] post = [6, 2, 1, 5, 4, 3, 7, 8] -> SAI | bản đúng -> hợp lệ


Vì sao sai: đoạn cây trái bị thiếu một nút (bị đẩy sang cây phải), nên đoạn chia sai và hai dãy không khớp. **Đúng:** `m = i - c + 1`.

### Lỗi 3 — Thiếu trường hợp cơ sở "một nút"

Khi `a == b` không có `preorder[a + 1]`: truy cập sẽ vượt mảng.

In [24]:
pre, post = counters["thiếu trường hợp một nút"]
print("pre =", pre, "post =", post, "->", attempt(wrong_no_base, pre, post), "| bản đúng ->", attempt(Solution().constructFromPrePost, pre, post))
assert attempt(wrong_no_base, [1], [1]) == "lỗi IndexError"

pre = [8, 7, 3, 4, 2, 6, 5, 1] post = [6, 2, 1, 5, 4, 3, 7, 8] -> lỗi IndexError | bản đúng -> hợp lệ


Vì sao sai: nút lá làm `a == b` và `preorder[a + 1]` không tồn tại (hoặc thuộc nhánh khác). **Đúng:** `if a == b: return root` trước khi tra `preorder[a + 1]`.

### Lỗi 4 — Coi phần tử kế tiếp của preorder là gốc cây con **phải**

Trong preorder, sau gốc là **cây con trái** (nếu có). Gán nó cho cây phải làm đảo vai trò và dãy hậu thứ tự không khớp (trừ trường hợp một con).

In [25]:
pre, post = counters["coi pre[a+1] là gốc cây phải"]
print("pre =", pre, "post =", post, "->", attempt(wrong_right_root, pre, post), "| bản đúng ->", attempt(Solution().constructFromPrePost, pre, post))
assert attempt(wrong_right_root, pre, post) != "hợp lệ"

pre = [8, 7, 3, 4, 2, 6, 5, 1] post = [6, 2, 1, 5, 4, 3, 7, 8] -> lỗi RecursionError | bản đúng -> hợp lệ


Vì sao sai: gốc cây phải đứng **cuối** đoạn `postorder` của gốc hiện tại (ngay trước nó), không phải ở vị trí `pre[a + 1]` đã cho. **Đúng:** coi `pre[a + 1]` là gốc cây trái, kết thúc đoạn trái ở `pos[pre[a + 1]]`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "dựng cây từ hai dãy duyệt" ⇒ xác định **gốc** từ một dãy và **kích thước cây con** từ dãy kia.
- **Preorder + inorder (105)** và **inorder + postorder (106)** cho cây **duy nhất**; **preorder + postorder (889)** thì **không**: nút chỉ một con là điểm mơ hồ.
- Kỹ thuật: bảng băm `giá trị -> vị trí`, đệ quy trên **chỉ số** (không cắt mảng để tránh `O(n^2)`), hoặc ngăn xếp / hai con trỏ đọc.
- Với bài có nhiều đáp án: kiểm bằng **`is_valid`**, không so cây cố định.
- Cùng họ: Construct Binary Tree from Preorder and Inorder Traversal (105), from Inorder and Postorder (106), Construct Binary Search Tree from Preorder (1008), Serialize and Deserialize Binary Tree (297).
- Test nên có: `n = 1`, `[1,2]/[2,1]` (hai đáp án), chuỗi trái/phải, cây đầy đủ, nhãn không theo thứ tự.

## 15. Biến thể

1. **Đếm số cây hợp lệ:** bằng `2^s`, với `s` là số nút có **đúng một con**; có thể tính `s` trực tiếp từ hai dãy (bài tập 5).
2. **Liệt kê mọi cây hợp lệ** (đã có `all_trees` ở mục 5).
3. **Preorder + postorder của cây đầy đủ** (mọi nút 0 hoặc 2 con): cây duy nhất.
4. Bài liên quan: 105, 106, 1008.

Công thức đếm: nút `v` có đúng một con ⇔ phần tử **kế tiếp** `v` trong preorder bằng phần tử **ngay trước** `v` trong postorder (khi đó là con duy nhất `c`: preorder `v, c` và postorder `c, v`). Kiểm chứng bằng oracle:

In [26]:
def count_valid_trees(pre: List[int], post: List[int]) -> int:
    n = len(pre)
    pos = {x: i for i, x in enumerate(post)}
    singles = 0
    for i in range(n - 1):
        v, nxt = pre[i], pre[i + 1]
        p = pos[v]
        if p > 0 and post[p - 1] == nxt:
            singles += 1
    return 2 ** singles

rng = random.Random(15)
for _ in range(400):
    n = rng.randint(1, 8)
    pre, post, t = random_shape_tree(rng, n)
    assert count_valid_trees(pre, post) == len(oracle_trees(pre, post)), (pre, post)
print("số cây của ví dụ 1:", count_valid_trees(*examples[0][:2]), "| của [1,2,3]/[3,2,1]:", count_valid_trees([1, 2, 3], [3, 2, 1]))
assert count_valid_trees([1, 2, 3], [3, 2, 1]) == 4

số cây của ví dụ 1: 1 | của [1,2,3]/[3,2,1]: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi kích thước | Quay lui + so tập | tệ nhất rất lớn | `O(n)` | Oracle, liệt kê mọi cây |
| 2. Chia để trị + `pos` | `pre[a+1]` cho kích thước cây trái | `O(n)` | `O(n)` | **Cách nộp chính** |
| 3. Ngăn xếp | Đóng nút khi `val == post[j]` | `O(n)` | `O(n)` | Không đệ quy |
| 4. Hai con trỏ đọc | Không cắt mảng, không băm | `O(n)` | `O(h)` | Gọn nhất |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
t = build_tree([1, 2, 3])
print(preorder_of(t), postorder_of(t))
pos = {x: i for i, x in enumerate([2, 3, 1])}
print(pos[3])
print(tree_to_list(build_tree([1, None, 2, 3])))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `preorder = [1,2,4,5,3]` và `postorder = [4,5,2,3,1]`: ghi danh sách theo mức của cây dựng bằng Cách 2.

In [28]:
# Bài 2: điền kết quả tính tay
hand_tree_list = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành chia để trị.

In [29]:
def exercise_3_dc(pre: List[int], post: List[int]) -> Optional[TreeNode]:
    pos = {x: i for i, x in enumerate(post)}

    def dfs(a: int, b: int, c: int, d: int) -> Optional[TreeNode]:
        if a > b:
            return None
        root = TreeNode(pre[a])
        if a == b:
            return root
        # TODO 1: i = vị trí của pre[a + 1] trong postorder; m = số nút cây con trái
        # TODO 2: dựng root.left và root.right bằng hai lời gọi đệ quy với các đoạn đúng
        return root

    return dfs(0, len(pre) - 1, 0, len(post) - 1)


# for pre, post, out in examples:
#     t = exercise_3_dc(pre, post)
#     assert is_valid(t, pre, post)

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_pointers(pre, post)` bằng hai con trỏ đọc.

In [30]:
def exercise_4_pointers(pre: List[int], post: List[int]) -> Optional[TreeNode]:
    # TODO
    pass


# for pre, post, out in examples:
#     t = exercise_4_pointers(pre, post)
#     assert is_valid(t, pre, post)

### Bài 5 — Mini challenge: đếm số cây hợp lệ (khó)

Viết `exercise_5_count(pre, post)` trả về **số cây khác nhau** có preorder `pre` và postorder `post`, **không dựng cây** (chỉ dùng hai dãy).

```python
exercise_5_count([1], [1])                          -> 1
exercise_5_count([1, 2], [2, 1])                    -> 2
exercise_5_count([1, 2, 3], [3, 2, 1])              -> 4
exercise_5_count([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1]) -> 1
```
Gợi ý: đếm nút có đúng một con; đáp án là `2` mũ số đó.

In [31]:
def exercise_5_count(pre: List[int], post: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_count([1], [1]) == 1
# assert exercise_5_count([1, 2], [2, 1]) == 2
# assert exercise_5_count([1, 2, 3], [3, 2, 1]) == 4
# assert exercise_5_count([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1]) == 1

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: preorder là gốc, trái, phải; postorder là trái, phải, gốc.
- Mức 2: cây `[1,2,3]`: gốc `1`, con trái `2`, con phải `3`; `pos` ánh xạ giá trị sang chỉ số trong `[2, 3, 1]`.
- Mức 3: `[1, 2, 3] [2, 3, 1]`, `1`, `[1, None, 2, 3]`.

**Bài 2**
- Mức 1: gốc `1`; phần tử kế tiếp `2` trong preorder.
- Mức 2: `2` ở chỉ số `2` của postorder, nên cây trái có `3` nút: `[2,4,5]` / `[4,5,2]`; còn lại là nút `3`.
- Mức 3: `[1, 2, 3, 4, 5]`.

**Bài 3**
- Mức 1: `i = pos[pre[a + 1]]`.
- Mức 2: `m = i - c + 1`.
- Mức 3: `root.left = dfs(a + 1, a + m, c, i)`; `root.right = dfs(a + m + 1, b, i + 1, d - 1)`.

**Bài 4**
- Mức 1: hai biến `i`, `j` dùng `nonlocal`.
- Mức 2: tạo `root = TreeNode(pre[i])`, `i += 1`.
- Mức 3: `if root.val != post[j]: root.left = build()`; lặp lại cho `right`; cuối cùng `j += 1`. Khi `root.val == post[j]` ngay sau khi tạo (lá) thì không dựng con nào.

**Bài 5**
- Mức 1: nút `v` có đúng một con `c` ⇔ preorder `v, c` và postorder `c, v`.
- Mức 2: với mỗi `i`, `v = pre[i]`, kiểm tra `post[pos[v] - 1] == pre[i + 1]`.
- Mức 3: `2 ** (số nút thỏa)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
t = build_tree([1, 2, 3])
print(preorder_of(t), postorder_of(t))
pos = {x: i for i, x in enumerate([2, 3, 1])}
print(pos[3])
print(tree_to_list(build_tree([1, None, 2, 3])))

assert (preorder_of(t), postorder_of(t)) == ([1, 2, 3], [2, 3, 1])
assert pos[3] == 1
assert tree_to_list(build_tree([1, None, 2, 3])) == [1, None, 2, 3]

[1, 2, 3] [2, 3, 1]
1
[1, None, 2, 3]


### Đáp án Bài 2

Gốc `1`; cây trái `[2,4,5]` / `[4,5,2]` (gốc `2`, con `4`, `5`); cây phải `[3]`. Danh sách theo mức: `[1, 2, 3, 4, 5]`.

In [33]:
pre, post = [1, 2, 4, 5, 3], [4, 5, 2, 3, 1]
hand_tree_list_solution = [1, 2, 3, 4, 5]
t = Solution().constructFromPrePost(pre, post)
assert tree_to_list(t) == hand_tree_list_solution
assert is_valid(t, pre, post)
assert len(oracle_trees(pre, post)) == 1            # cây duy nhất
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_dc_solution(pre: List[int], post: List[int]) -> Optional[TreeNode]:
    pos = {x: i for i, x in enumerate(post)}

    def dfs(a: int, b: int, c: int, d: int) -> Optional[TreeNode]:
        if a > b:
            return None
        root = TreeNode(pre[a])
        if a == b:
            return root
        i = pos[pre[a + 1]]
        m = i - c + 1
        root.left = dfs(a + 1, a + m, c, i)
        root.right = dfs(a + m + 1, b, i + 1, d - 1)
        return root

    return dfs(0, len(pre) - 1, 0, len(post) - 1)

for pre, post, out in examples:
    t = exercise_3_dc_solution(pre, post)
    assert is_valid(t, pre, post)
rng = random.Random(3)
for _ in range(300):
    n = rng.randint(1, 8)
    pre, post, _t = random_shape_tree(rng, n)
    t = exercise_3_dc_solution(pre, post)
    assert is_valid(t, pre, post) and to_labeled(t) in oracle_trees(pre, post)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_pointers_solution(pre: List[int], post: List[int]) -> Optional[TreeNode]:
    i = j = 0

    def build() -> TreeNode:
        nonlocal i, j
        root = TreeNode(pre[i])
        i += 1
        if root.val != post[j]:
            root.left = build()
        if root.val != post[j]:
            root.right = build()
        j += 1
        return root

    return build()

for pre, post, out in examples:
    assert is_valid(exercise_4_pointers_solution(pre, post), pre, post)
rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 8)
    pre, post, _t = random_shape_tree(rng, n)
    t = exercise_4_pointers_solution(pre, post)
    assert is_valid(t, pre, post) and to_labeled(t) in oracle_trees(pre, post)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_count_solution(pre: List[int], post: List[int]) -> int:
    pos = {x: i for i, x in enumerate(post)}
    singles = 0
    for i in range(len(pre) - 1):
        p = pos[pre[i]]
        if p > 0 and post[p - 1] == pre[i + 1]:
            singles += 1
    return 2 ** singles

assert exercise_5_count_solution([1], [1]) == 1
assert exercise_5_count_solution([1, 2], [2, 1]) == 2
assert exercise_5_count_solution([1, 2, 3], [3, 2, 1]) == 4
assert exercise_5_count_solution([1, 2, 4, 5, 3, 6, 7], [4, 5, 2, 6, 7, 3, 1]) == 1
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 8)
    pre, post, _t = random_shape_tree(rng, n)
    assert exercise_5_count_solution(pre, post) == len(oracle_trees(pre, post)), (pre, post)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (chia để trị trên đoạn, bảng băm vị trí trong postorder)

```python
class Solution:
    def constructFromPrePost(self, preorder: List[int], postorder: List[int]) -> Optional[TreeNode]:
        pos = {x: i for i, x in enumerate(postorder)}          # vị trí trong postorder

        def dfs(a: int, b: int, c: int, d: int) -> Optional[TreeNode]:
            if a > b:                                           # đoạn rỗng
                return None
            root = TreeNode(preorder[a])                        # gốc = đầu preorder
            if a == b:                                          # một nút: không có preorder[a + 1]
                return root
            i = pos[preorder[a + 1]]                            # gốc cây con trái (quy ước)
            m = i - c + 1                                       # số nút cây con trái
            root.left = dfs(a + 1, a + m, c, i)
            root.right = dfs(a + m + 1, b, i + 1, d - 1)
            return root

        return dfs(0, len(preorder) - 1, 0, len(postorder) - 1)
```

Công thức nhớ nhanh: gốc = `pre[a]`; gốc trái = `pre[a + 1]`; `m = pos[pre[a + 1]] - c + 1`.

### Checklist trước khi submit

- [ ] Có trường hợp cơ sở `a > b` (rỗng) và `a == b` (một nút)?
- [ ] `m = i - c + 1` (tính cả gốc cây trái)?
- [ ] Đoạn phải trong postorder là `[i + 1, d - 1]` (bỏ gốc cuối)?
- [ ] Dùng `pos` (băm) thay `list.index` để giữ `O(n)`?
- [ ] Kiểm thử bằng `is_valid` (nhiều đáp án), không so cây cố định?
- [ ] Đã thử: `n = 1`, `[1,2]/[2,1]`, chuỗi trái/phải, cây đầy đủ, nhãn không theo thứ tự?
- [ ] Tên method đúng `constructFromPrePost(self, preorder, postorder)` và trả `TreeNode`?